# $\color{teal}{\text{Shipping a Change}}$

At this point, we've seen how Need to Know goes from a pull request to a running release: GitHub checks every change, builds the image once, names it, and pushes it to ghcr.io, and one script deploys it on our instance. In this notebook, we'll click through the same steps ourselves, in the same order as the lecture: the checks, the names, the deploy log, the cache headers, and the four releases that changed our stored data.

Run it from the repository on branch `example-l8`, after `uv sync --extra dev`. Everything here runs on your own machine (no Docker, and no instance needed).

In [ ]:
import json
import logging
import sqlite3
import subprocess
import tomllib
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")
logging.disable(logging.INFO)          # keep the app's request log out of the way

ROOT = Path.cwd() if (Path.cwd() / "pyproject.toml").exists() else Path.cwd().parent
print(sorted(p.name for p in (ROOT / ".github" / "workflows").iterdir()))

"""
['ci.yml', 'release.yml']
"""

## $\color{teal}{\text{What does our workflow tell GitHub to do?}}$

Our **workflow** is one YAML file, `.github/workflows/ci.yml`, and GitHub runs it on every push and pull request. Since it's YAML, we can load it like any other data and ask it questions. Here are its triggers, and for each job, what it waits for and what it's allowed to do.

In [ ]:
import yaml

ci = yaml.safe_load((ROOT / ".github" / "workflows" / "ci.yml").read_text())
print("triggers:", list(ci[True]))          # YAML reads the key `on` as the boolean True
print("default permissions:", ci["permissions"])
for name, job in ci["jobs"].items():
    print(f"{name:8} needs={job.get('needs', '-')!s:5} if={job.get('if', '-')}  permissions={job.get('permissions', 'default')}")

"""
triggers: ['push', 'pull_request', 'workflow_dispatch']
default permissions: {'contents': 'read'}
test     needs=-     if=-  permissions=default
publish  needs=test  if=github.event_name == 'push'  permissions={'contents': 'read', 'packages': 'write'}
"""

The `test` job is the **continuous integration** half: it runs our checks. The `publish` job is the **continuous delivery** half: it builds the image, smoke tests it, and pushes it, but only after `test` passes, and only on a push.

**Q. Only one job can push to ghcr.io. Which line of the output tells us, and what would happen if we deleted it?**

<details>
<summary>Click here for an answer</summary>


The `publish` job's `permissions={'contents': 'read', 'packages': 'write'}`. Without it, the job gets the default from the top of the file, `contents: read`, and `docker push` is refused. The tests would still pass; we'd just stop publishing images.

</details>

In [ ]:
test = ci["jobs"]["test"]
print("matrix:", test["strategy"]["matrix"])
print("concurrency:", ci["concurrency"])
for step in test["steps"]:
    print(" ", step.get("uses") or step.get("run"))

"""
matrix: {'python': ['3.12', '3.13']}
concurrency: {'group': '${{ github.workflow }}-${{ github.ref }}', 'cancel-in-progress': "${{ github.event_name == 'pull_request' }}"}
  actions/checkout@v4
  astral-sh/setup-uv@v6
  uv sync --frozen --extra dev
  uv run ruff check src tests
  uv run pytest -q
"""

Now let's run the same checks CI runs, on our own machine. These are the last two steps above.

In [ ]:
import os
import re

for cmd in (["uv", "run", "ruff", "check", "src", "tests"], ["uv", "run", "pytest", "-q", "--color=no"]):
    out = subprocess.run(cmd, cwd=ROOT, capture_output=True, text=True, env={**os.environ, "NO_COLOR": "1"})
    last = out.stdout.strip().splitlines()[-1]
    print("$", " ".join(cmd)); print(re.sub(r"\[[0-9;]*m", "", last))   # drop the terminal colors

"""
$ uv run ruff check src tests
All checks passed!
$ uv run pytest -q --color=no
25 passed, 1 warning in 0.82s
"""

**Q. CI runs these exact commands on a fresh machine every time. Why not just trust that they passed on our laptop?**

<details>
<summary>Click here for an answer</summary>


Because our laptop isn't a fresh machine. It may have a package installed that isn't in `uv.lock`, a file that isn't committed, or an old `.env`. A runner starts from nothing, so a green check there means the repository itself works, which is CI's answer to "but the tests passed on my laptop".

</details>

## $\color{teal}{\text{How do we name an image?}}$

CI tags every image with the commit it was built from, using the first 7 characters of the commit hash. Let's compute the name CI would give our current commit (yours will show whatever commit you have checked out).

In [ ]:
sha = subprocess.run(["git", "rev-parse", "--short=7", "HEAD"], cwd=ROOT, capture_output=True, text=True).stdout.strip()
print(f"ghcr.io/trevoradriaanse/needtoknow:sha-{sha}")

A **release** gives one of those images a second name, a version. The version lives in one place, `pyproject.toml`, and `release.yml` refuses a tag that doesn't match it.

In [ ]:
version = tomllib.loads((ROOT / "pyproject.toml").read_text())["project"]["version"]
release = yaml.safe_load((ROOT / ".github" / "workflows" / "release.yml").read_text())
print("version in pyproject.toml:", version)
print("release.yml runs on tags:", release[True]["push"]["tags"])
print("so the release tag must be:", f"v{version}")

"""
version in pyproject.toml: 0.6.0
release.yml runs on tags: ['v*']
so the release tag must be: v0.6.0
"""

Versions follow **semantic versioning**, MAJOR.MINOR.PATCH: a fix raises PATCH, and a new feature raises MINOR. The release job also copies that version's section of `CHANGELOG.md` into its summary. Here's the same thing in Python.

In [ ]:
import re

def notes(version):
    text = (ROOT / "CHANGELOG.md").read_text()
    sections = re.split(r"^## ", text, flags=re.M)
    return next(s for s in sections if s.startswith(f"[{version}]"))

print(notes("0.4.0"))

"""
[0.4.0] - 2026-10-15

### Added
- Counts per detector (`by_detector` in `/api/stats`), so detectors can be compared after a switch. Migration
  `0002_usage_by_detector.sql` adds a table and copies the counts so far into it as `regex`.

### Note for whoever deploys
- This version writes both the old table and the new one, so rolling back to v0.3.0 loses nothing.
"""

**Q. Issue #8 adds support for .docx files. If Need to Know is at 0.6.1, what's the next version?**

<details>
<summary>Click here for an answer</summary>


0.7.0. Supporting a new file type is a new feature that breaks nothing, so MINOR goes up and PATCH goes back to 0.

</details>

## $\color{teal}{\text{What can deploy.log tell us?}}$

Every time we run `scripts/deploy.sh`, it writes one line to `deploy.log`: when, which account, the image, its digest, and the commit. Here are the real lines from our rehearsal on the class instance.

In [ ]:
LOG = """\
2026-10-15T23:04:21Z exouser deployed ghcr.io/trevoradriaanse/needtoknow:v0.2.1 ghcr.io/trevoradriaanse/needtoknow@sha256:fd18561ca954f51364d48dbb0fa234c33724f2e3cf997dc9be13802e1171152a commit 91e28e8
2026-10-15T23:04:29Z exouser deployed ghcr.io/trevoradriaanse/needtoknow:sha-bed9ddd ghcr.io/trevoradriaanse/needtoknow@sha256:7a3863305f150124aac869f8ef35f9fe43b3114d435e3100facc975db5715214 commit bed9ddd
2026-10-15T23:04:33Z exouser deployed ghcr.io/trevoradriaanse/needtoknow:v0.2.1 ghcr.io/trevoradriaanse/needtoknow@sha256:fd18561ca954f51364d48dbb0fa234c33724f2e3cf997dc9be13802e1171152a commit 91e28e8
2026-10-15T23:04:33Z exouser FAILED ghcr.io/trevoradriaanse/needtoknow:v9.9.9: could not pull it
2026-10-15T23:04:53Z exouser deployed ghcr.io/trevoradriaanse/needtoknow:v0.3.0 ghcr.io/trevoradriaanse/needtoknow@sha256:8701a8eaeaa54126d49a1fda273cd76f45d5c3b03176672383ac3c23c376e3ec commit 32e9a45
2026-10-15T23:05:11Z exouser deployed ghcr.io/trevoradriaanse/needtoknow:v0.4.0 ghcr.io/trevoradriaanse/needtoknow@sha256:81d8262ad1261df0da49f376f75093dffce72c45a5505c146b517f0a28cdcc58 commit 30f01ce"""

deployed = [line.split() for line in LOG.splitlines() if " deployed " in line]
print("running now:   ", deployed[-1][3].split(":")[-1], "commit", deployed[-1][-1])
print("roll back to:  ", deployed[-2][3].split(":")[-1], "commit", deployed[-2][-1])

"""
running now:    v0.4.0 commit 30f01ce
roll back to:   v0.3.0 commit 32e9a45
"""

**Q. The failed v9.9.9 line isn't a candidate for a rollback. Why is it still worth keeping in the log?**

<details>
<summary>Click here for an answer</summary>


It tells the next person what was tried and when, and the exit code it matches (2, the pull failed). Remember, a failed pull left `.env` and the running app as they were, so nothing needs undoing.

</details>

`scripts/prune.sh` uses the same lines to decide which images we can delete: it keeps the one running and the last three deployed. Here's that rule in Python.

In [ ]:
keep = []
for fields in reversed(deployed):
    if fields[3] not in keep:
        keep.append(fields[3])
print("keep:", [k.split(":")[-1] for k in keep[:3]])

"""
keep: ['v0.4.0', 'v0.3.0', 'v0.2.1']
"""

## $\color{teal}{\text{Which response may be kept, and by whom?}}$

When we open a web page twice, the browser doesn't download everything again: it keeps copies of what it already has in its **cache**. Ten people opening Need to Know means ten browsers and ten caches, each on its own laptop, and nobody sees anyone else's. A **CDN** (content delivery network) is different: it's one shared cache between us and everyone, so it may only keep what's the same for everyone.

Who may keep a copy, and for how long, is up to us. We say so in the **Cache-Control** header, which our app sets for every response in one middleware in `api/main.py`. FastAPI's `TestClient` calls the app in-process, so we can check every rule without starting a server.

In [ ]:
from fastapi.testclient import TestClient

from needtoknow.api.main import app
from needtoknow.config import settings

lease = (ROOT / "scripts" / "smoke-lease.pdf").read_bytes()
with TestClient(app) as client:     # `with` runs the app's startup, which loads the detector
    for label, r in [("GET /", client.get("/")),
                     (f"GET /static/app.js?v={settings.commit}", client.get(f"/static/app.js?v={settings.commit}")),
                     ("GET /static/app.js?v=bed9ddd", client.get("/static/app.js?v=bed9ddd")),
                     ("POST /api/detect", client.post("/api/detect", files={"file": ("lease.pdf", lease, "application/pdf")})),
                     ("GET /api/stats", client.get("/api/stats"))]:
        print(f"{label:30} {r.status_code}  {r.headers.get('cache-control')}")

"""
GET /                          200  no-cache
GET /static/app.js?v=dev       200  public, max-age=31536000, immutable
GET /static/app.js?v=bed9ddd   200  no-cache
POST /api/detect               200  no-store
GET /api/stats                 200  public, max-age=60
"""

Running from source, the commit is `dev`, so the current script is `app.js?v=dev`. In the image, it's the seven characters CI passed in.

**Q. Why may `/api/stats` be kept for a minute, but `/api/detect` never?**

<details>
<summary>Click here for an answer</summary>


Whose answer is it? `/api/stats` is the same totals for everyone, so a browser or a CDN may keep it for a minute. `/api/detect` is one person's findings from their own document, so **no browser and no CDN may keep a copy at all**, which is what `no-store` says.

</details>

`no-cache` doesn't mean "don't keep it": it means keep it, but ask us before using it. Asking is cheap thanks to the **ETag**, a fingerprint of the page. The browser sends it back, and if the page hasn't changed, we answer `304 Not Modified` with no page at all.

In [ ]:
import hashlib

with TestClient(app) as client:
    page = client.get("/")
    again = client.get("/", headers={"If-None-Match": page.headers["etag"]})
print("ETag:", page.headers["etag"], " bytes:", len(page.content))
print("our own hash of the page:", '"' + hashlib.sha256(page.content).hexdigest()[:16] + '"')
print("asking again with that ETag:", again.status_code, " bytes:", len(again.content))

"""
ETag: "c02e3705180e8ccf"  bytes: 2986
our own hash of the page: "c02e3705180e8ccf"
asking again with that ETag: 304  bytes: 0
"""

**Q. Someone loads our page from the old version just as we deploy, and their script request arrives a second later, asking for `app.js?v=bed9ddd`. What does our app answer, and why does it matter?**

<details>
<summary>Click here for an answer</summary>


It sends the new `app.js`, since that's the only one it has, but marked `no-cache` (the third line of the earlier output). If it said keep this for a year, the new code would be filed under the old address, and the next time we rolled back to bed9ddd, browsers would run the wrong script from their cache.

</details>

## $\color{teal}{\text{How do we change stored data safely?}}$

Need to Know keeps one kind of data: **usage counts**, how many documents were checked and how many items were found each day (never the documents themselves), in a small SQLite database. The shape of a database, its tables and columns, is its **schema**, and a **migration** is a numbered file that changes the schema once, in order.

Rolling back an image doesn't roll back our data. So we change the schema the way we'd change our phone number: get the new one while the old one still works (**expand**), switch everyone over, and only then cancel the old one (**contract**), with a backup before the step that can't be undone. Let's walk through our four releases on a **temporary database**, using the same migration files the image runs.

In [ ]:
import tempfile

from needtoknow import migrate, store

tmp = Path(tempfile.mkdtemp())
db = str(tmp / "needtoknow.db")
everything = migrate.available()
print([name for _, name, _ in everything])

migrate.available = lambda: everything[:1]          # v0.3.0: the first table
migrate.up(db)
con = sqlite3.connect(db)
with con:
    con.execute("INSERT INTO usage VALUES ('2026-10-14', 7, 30), ('2026-10-15', 2, 9)")   # what v0.3.0 counted
print(con.execute("SELECT * FROM usage").fetchall())

"""
['0001_usage.sql', '0002_usage_by_detector.sql', '0003_drop_usage.sql']
applied 0001_usage.sql
schema is at 1
[('2026-10-14', 7, 30), ('2026-10-15', 2, 9)]
"""

In [ ]:
migrate.available = lambda: everything[:2]          # v0.4.0: expand
migrate.up(db)
print(con.execute("SELECT * FROM usage_by_detector").fetchall())
print("old total:", con.execute("SELECT SUM(documents) FROM usage").fetchone()[0],
      " new total:", con.execute("SELECT SUM(documents) FROM usage_by_detector").fetchone()[0])
print(sorted(p.name for p in (tmp / "backups").iterdir()))

"""
backed up to /tmp/tmp31faqok9/backups/needtoknow-20261020T000141Z-before-0002.db
applied 0002_usage_by_detector.sql
schema is at 2
[('2026-10-14', 'regex', 7, 30), ('2026-10-15', 'regex', 2, 9)]
old total: 9  new total: 9
['needtoknow-20261020T000141Z-before-0002.db']
"""

**Q. Why does the backup's name say `before-0002`, and when would we need it?**

<details>
<summary>Click here for an answer</summary>


It's the database exactly as it was before migration 0002 ran, so it matches what v0.3.0 expects. We'd need it only if 0002 went wrong in a way a rollback can't fix. After an expand step that's rare, since v0.3.0 still finds its table and keeps working.

</details>

v0.5.0 stops using the old table, with no migration at all. Then v0.6.0 drops it.

In [ ]:
migrate.available = lambda: everything                # v0.6.0: contract
migrate.up(db)
migrate.status(db)
print(sorted(p.name for p in (tmp / "backups").iterdir()))

"""
backed up to /tmp/tmp31faqok9/backups/needtoknow-20261020T000141Z-before-0003.db
applied 0003_drop_usage.sql
schema is at 3
applied  0001_usage.sql
applied  0002_usage_by_detector.sql
applied  0003_drop_usage.sql
schema is at 3
['needtoknow-20261020T000141Z-before-0002.db', 'needtoknow-20261020T000141Z-before-0003.db']
"""

Now suppose someone rolls the image back to v0.4.0. Its code still writes to the old table. Here's that write, by hand:

In [ ]:
try:
    with con:
        con.execute("INSERT INTO usage (day, documents, items) VALUES ('2026-10-16', 1, 6)")
except sqlite3.OperationalError as exc:
    print("v0.4.0 would fail with:", exc)

"""
v0.4.0 would fail with: no such table: usage
"""

**Q. v0.4.0's `/ready` still says ready on this database. Why doesn't it notice, and what does?**

<details>
<summary>Click here for an answer</summary>


`/ready` checks that the schema is at least the version the code needs: v0.4.0 needs 2 and finds 3, which is allowed, because old code runs on a newer schema during every deploy. The failure only shows up when it writes, and recording a count never fails the user's request. **`scripts/smoke.sh` notices**, because it checks that the document it sent was counted.

</details>

To go back after dropping a table, we have to restore the backup (the app would be stopped first):

In [ ]:
backup = sorted((tmp / "backups").glob("*before-0003.db"))[-1]
con.close()
Path(db).write_bytes(backup.read_bytes())
con = sqlite3.connect(db)
print("schema:", store.schema_version(db), " tables:",
      [r[0] for r in con.execute("SELECT name FROM sqlite_master WHERE type = 'table' ORDER BY name")])

"""
schema: 2  tables: ['schema_migrations', 'usage', 'usage_by_detector']
"""

**Q. We restored the backup. What did we lose?**

<details>
<summary>Click here for an answer</summary>


Every count recorded after the backup was taken, that is, after v0.6.0 was deployed. That's the price of undoing a contract step, and it's why the runbook marks it as the step that can't be undone.

</details>

## $\color{teal}{\text{Take-home message}}$

We read our workflow as data and ran the same checks CI runs, named an image and a release the way our pipeline does, used `deploy.log` to find what to roll back to, checked every cache header, and took a real database through expand and contract, including a rollback that fails and a restore that loses data. These are the same files the image runs on the instance through `scripts/deploy.sh`.

**You've shipped a change, and a change to stored data!**